# Clase 033 — Polars: DataFrames modernos

Comparamos Polars vs pandas sobre un dataset sintético generado en disco. Requiere: `pip install polars pandas pyarrow numpy`.

In [ ]:
import time, tempfile, os
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import polars as pl
except ImportError as e:
    raise ImportError('Instalá polars: pip install polars pyarrow') from e

print(f'polars {pl.__version__} | pandas {pd.__version__} | numpy {np.__version__}')

## 1. Generamos dataset sintético (~50 MB CSV en disco)

In [ ]:
rng = np.random.default_rng(42)
N = 2_000_000

tmpdir = Path(tempfile.mkdtemp(prefix='polars_demo_'))
csv_path = tmpdir / 'ventas.csv'

categorias = np.array(['A', 'B', 'C', 'D', 'E'])
regiones = np.array(['Norte', 'Sur', 'Este', 'Oeste'])

df_gen = pd.DataFrame({
    'id': np.arange(N),
    'categoria': rng.choice(categorias, N),
    'region': rng.choice(regiones, N),
    'precio': rng.gamma(2.0, 50.0, N).round(2),
    'cantidad': rng.integers(1, 20, N),
    'descuento': rng.uniform(0, 0.3, N).round(3),
})
df_gen.to_csv(csv_path, index=False)
size_mb = csv_path.stat().st_size / 1e6
print(f'CSV escrito: {csv_path} ({size_mb:.1f} MB, {N:,} filas)')

## 2. Benchmark de lectura: pandas vs Polars eager

In [ ]:
t0 = time.perf_counter()
pdf = pd.read_csv(csv_path)
t_pd = time.perf_counter() - t0

t0 = time.perf_counter()
pldf = pl.read_csv(csv_path)
t_pl = time.perf_counter() - t0

print(f'pandas.read_csv : {t_pd:.3f} s')
print(f'polars.read_csv : {t_pl:.3f} s')
print(f'speedup polars  : {t_pd / t_pl:.2f}x')

## 3. API de expresiones — `pl.col`, `when/then/otherwise`

In [ ]:
result = pldf.with_columns([
    (pl.col('precio') * pl.col('cantidad') * (1 - pl.col('descuento'))).alias('total'),
    pl.when(pl.col('precio') > 100).then(pl.lit('caro'))
      .when(pl.col('precio') > 50).then(pl.lit('medio'))
      .otherwise(pl.lit('barato')).alias('rango'),
])
print(result.select(['precio', 'cantidad', 'descuento', 'total', 'rango']).head(5))

## 4. group_by + agregaciones — Polars vs pandas

In [ ]:
t0 = time.perf_counter()
g_pd = pdf.groupby(['categoria', 'region']).agg(
    precio_medio=('precio', 'mean'),
    cant_total=('cantidad', 'sum'),
    n=('id', 'count'),
)
t_pd_g = time.perf_counter() - t0

t0 = time.perf_counter()
g_pl = pldf.group_by(['categoria', 'region']).agg([
    pl.col('precio').mean().alias('precio_medio'),
    pl.col('cantidad').sum().alias('cant_total'),
    pl.len().alias('n'),
]).sort(['categoria', 'region'])
t_pl_g = time.perf_counter() - t0

print(f'pandas groupby: {t_pd_g:.3f}s | polars group_by: {t_pl_g:.3f}s | speedup {t_pd_g/t_pl_g:.2f}x')
print(g_pl.head(8))

## 5. LazyFrame + query plan (`scan_csv` + `.explain()`)

In [ ]:
lf = (
    pl.scan_csv(csv_path)
      .filter(pl.col('precio') > 100)
      .group_by('categoria')
      .agg([pl.col('cantidad').sum().alias('total_cant'),
            pl.col('precio').mean().alias('precio_medio')])
      .sort('total_cant', descending=True)
)
print('=== Query plan optimizado ===')
print(lf.explain())

t0 = time.perf_counter()
out = lf.collect()
t_lazy = time.perf_counter() - t0
print(f'\nLazy collect: {t_lazy:.3f}s')
print(out)

## 6. Joins

In [ ]:
dim_categoria = pl.DataFrame({
    'categoria': ['A', 'B', 'C', 'D', 'E'],
    'familia': ['Alimentos', 'Bebidas', 'Limpieza', 'Hogar', 'Otros'],
    'margen': [0.20, 0.35, 0.40, 0.25, 0.15],
})

joined = pldf.join(dim_categoria, on='categoria', how='left')
print(joined.select(['id', 'categoria', 'familia', 'margen', 'precio']).head(5))

## 7. Window functions (rank, cumulative, over)

In [ ]:
window = pldf.with_columns([
    pl.col('precio').mean().over('categoria').alias('precio_medio_cat'),
    pl.col('precio').rank(method='dense', descending=True).over('region').alias('rank_en_region'),
])
print(window.select(['categoria', 'region', 'precio', 'precio_medio_cat', 'rank_en_region']).head(8))

## 8. Arrow zero-copy: Polars ↔ pandas ↔ pyarrow

In [ ]:
small = pldf.head(100_000)

t0 = time.perf_counter()
arrow_table = small.to_arrow()
t_arrow = time.perf_counter() - t0

t0 = time.perf_counter()
pdf_from_arrow = arrow_table.to_pandas()
t_to_pd = time.perf_counter() - t0

t0 = time.perf_counter()
back_to_pl = pl.from_pandas(pdf_from_arrow)
t_back = time.perf_counter() - t0

print(f'polars → arrow : {t_arrow*1000:.2f} ms')
print(f'arrow → pandas : {t_to_pd*1000:.2f} ms')
print(f'pandas → polars: {t_back*1000:.2f} ms')
print(f'arrow table: {arrow_table.num_rows:,} filas, {arrow_table.num_columns} cols')

## 9. Cleanup

In [ ]:
import shutil
shutil.rmtree(tmpdir, ignore_errors=True)
print(f'limpieza ok: {tmpdir} eliminado')

## Ejercicios

1. Repetí el benchmark con `N = 10_000_000` y reportá speedup.
2. Usá `lf.collect(streaming=True)` para un dataset mayor que RAM.
3. Reemplazá el join con `how='inner'` y verificá conteos.
4. Agregá una expresión window de moving average sobre `precio` ordenado por `id`.
5. Convertí a Parquet con `pldf.write_parquet('out.parquet', compression='zstd')` y comparalo en tamaño con el CSV original.